# Liputan6 -> AMR (Indonesian -> Indonesian AMR)

Parses Liputan6 sentences into AMR graphs using **Abdi's ID->ID parser**:
`abdiharyadi/mbart-en-id-smaller-indo-amr-parsing-translated-nafkhan`
(Smatch 0.8299).

**No translation step.** The model takes plain Indonesian. Verified from the
model repo's own `dummy_input.json`, whose reference input is:

```
Kami...</s> <AMR> <mask> </AMR> <pad><pad><pad>
```

i.e. `<indonesian text> </s> <AMR> <mask> </AMR>` - no `id_ID`, no `en_XX`,
no English. Section 4 below *verifies* this by reproducing those exact token ids.

## Setup
1. **Add Input:** `amr-code-modules` (the `common/` + `model_interface/` folders)
2. **Add Input:** `liputan6-data` (`analysis_data.csv`)
3. **Accelerator:** GPU **T4**
4. **Internet: ON** - the model is downloaded from HuggingFace
5. **Persistence: Files only**

## How to run
Click **Save Version -> Save & Run All (Commit)** and close the browser: Kaggle
runs it headless and saves the output automatically. Running the cells
interactively also works, but then `/kaggle/working` is lost when the session
ends unless you click Save Version yourself.

(An earlier version built a Python 3.10 conda env and required a manual kernel
switch. Kaggle's image has no `conda` on PATH and its default Python 3.12 runs
the repo code fine, so that step is gone.)

# 1. Install dependencies

In [ ]:
# Kaggle's default Python (3.12) runs the repo code fine - verified by importing
# model_interface/common below. No conda env and no kernel switch needed.
#
# transformers is pinned to the version this model was trained with (see its
# README). torch is NOT reinstalled: Kaggle's build already has working CUDA and
# swapping it risks breaking that.
!pip install --quiet "transformers==4.44.0" "penman>=1.1.0" sentencepiece sacremoses regex networkx huggingface_hub

import sys, torch, transformers
print("Python      :", sys.version.split()[0])
print("transformers:", transformers.__version__)
print("torch       :", torch.__version__, "| CUDA:", torch.cuda.is_available())

Dependencies land in the running kernel, so just continue to Cell 2 once the install finishes.

# 2. Imports & paths

In [ ]:
import os, sys, json, shutil, time, torch, penman, pandas as pd
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# ============================================================
# PATHS - Kaggle mounts datasets as
#   /kaggle/input/datasets/<username>/<dataset-slug>/
# ============================================================
BASE              = "/kaggle/input/datasets/fedrianzdharma"
CODE_MODULES_PATH = f"{BASE}/amr-code-modules"
DATA_PATH         = f"{BASE}/liputan6-data"

# Model is pulled straight from HuggingFace (needs Internet ON) - no dataset upload.
MODEL_ID          = "abdiharyadi/mbart-en-id-smaller-indo-amr-parsing-translated-nafkhan"

OUTPUT_DIR        = "/kaggle/working/amr_graphs"

# CROSS-SESSION RESUME: save each run's output as a dataset named
# "liputan6-amr-graphs", then it mounts here. Use "" on the FIRST run.
PREV_DIR          = f"{BASE}/liputan6-amr-graphs/amr_graphs"
# PREV_DIR = ""   # <- first run

# ============================================================
# SCOPE - assignment target is ~10rb (10,000) TRAINING documents.
#   SPLIT     : which Liputan6 split to parse ("train"/"test"/None=all).
#               MUST be set - the csv is sorted with test BEFORE train, so
#               taking "the first N rows" silently gives you the test set.
#   DOC_LIMIT : cap on DOCUMENTS parsed (all sentences of a doc kept
#               together). None = no cap.
# ============================================================
SPLIT         = "train"
#   SENT_LIMIT: cap on SENTENCES. Documents are taken whole, in order, until
#               the budget is reached - never a partial document, because the
#               GNN needs every sentence of a doc to build its graph.
#   DOC_LIMIT : cap on documents. Applied first if both are set. None = no cap.
SENT_LIMIT    = 10000
DOC_LIMIT     = None

# ============================================================
# GENERATION / SPEED - a T4 is slow at the defaults below, and Liputan6
# sentences are far longer than the model's tiny reference examples.
#   USE_FP16   : halves memory and roughly doubles throughput on a T4.
#   BATCH_SIZE : the big win. Inputs are right-padded exactly as in the
#                checkpoint's reference, so batching is safe.
#   NUM_BEAMS  : 5 is the checkpoint's own default. 3 is ~40% cheaper,
#                1 (greedy) is ~5x cheaper but lower quality.
#   MAX_GEN_LEN: cap on generated AMR tokens. The model card reports a mean
#                gen_len of 27.5, so 512 only bounds runaway decodes.
# ============================================================
USE_FP16      = True
BATCH_SIZE    = 8
NUM_BEAMS     = 5
MAX_GEN_LEN   = 512

# Safety rails so an interactive session ends cleanly and you can Save Version.
MAX_NEW       = 40000     # stop after this many NEW graphs this run (None = no cap)
TIME_BUDGET_H = 8.0

os.makedirs(OUTPUT_DIR, exist_ok=True)
if CODE_MODULES_PATH not in sys.path:
    sys.path.insert(0, CODE_MODULES_PATH)

from transformers import AutoConfig
from model_interface.modeling_bart import MBartForConditionalGeneration
from model_interface.tokenization_bart import AMRBartTokenizer
from common.postprocessing import ParsedStatus

print("Python :", sys.version.split()[0])
print("Torch  :", torch.__version__, "| CUDA:", torch.cuda.is_available())
for name, p in [("code modules", CODE_MODULES_PATH), ("data", DATA_PATH),
                ("prev output", PREV_DIR)]:
    print(f"{name:13s} exists: {os.path.exists(p) if p else False}  ({p})")

# 3. Load model & tokenizer (from HuggingFace)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

amr_config    = AutoConfig.from_pretrained(MODEL_ID)
amr_tokenizer = AMRBartTokenizer.from_pretrained(MODEL_ID, use_fast=False)
amr_model     = MBartForConditionalGeneration.from_pretrained(MODEL_ID, config=amr_config)

# DO NOT call amr_model.resize_token_embeddings(len(amr_tokenizer)) here.
# AMRBartTokenizer keeps its own .vocab dict and its __len__ reports 0, so that
# call - which upstream uses when ADDING amr tokens to a base model - silently
# resizes the embedding matrix to torch.Size([0, 1024]) and destroys the model.
# This checkpoint already ships vocab_size 38025 with the AMR tokens included,
# so there is nothing to resize.
emb_rows = amr_model.get_input_embeddings().weight.shape[0]
assert emb_rows == amr_config.vocab_size, (
    f"embedding has {emb_rows} rows, expected {amr_config.vocab_size} - "
    "reload the model and do not resize its embeddings")

if USE_FP16 and torch.cuda.is_available():
    amr_model = amr_model.half()
amr_model = amr_model.to(device)
amr_model.eval()

print("Device     :", amr_model.device)
print("vocab_size :", amr_config.vocab_size)
print("embeddings :", tuple(amr_model.get_input_embeddings().weight.shape))
# Trust amr_tokenizer.vocab, not the tokenizer's special-token attributes:
# vocab['<mask>'] is 35107 while amr_tokenizer.mask_token_id reports 4.
print("<AMR>      :", amr_tokenizer.vocab["<AMR>"],
      "| </AMR>:", amr_tokenizer.vocab["</AMR>"],
      "| <mask>:", amr_tokenizer.vocab["<mask>"])

# 4. VERIFY the input format

The model repo ships `dummy_input.json` with reference `input_ids` for five
Indonesian sentences. We rebuild those inputs and assert the ids match exactly.

This is not ceremony - it caught a real bug. The trailing special tokens are
taken from the reference rather than from the tokenizer, because this
checkpoint's `<mask>` is **35107** while `amr_tokenizer.mask_token_id` reports
**4**. Trusting the attribute produces a well-formed but wrong input, and the
model would emit nonsense for every sentence with nothing obviously broken.

Expected suffix: `</s> <AMR> <mask> </AMR>` = `[2, 38023, 35107, 38024]`.

The cell asserts, so a mismatch stops the notebook before it burns GPU time.

In [ ]:
import ast
from huggingface_hub import hf_hub_download

# The repo stores input_ids as a STRING holding a python list literal, not as
# JSON arrays - hence literal_eval.
dummy_path = hf_hub_download(repo_id=MODEL_ID, filename="dummy_input.json")
dummy      = json.load(open(dummy_path, encoding="utf-8"))
ref_ids    = ast.literal_eval(dummy["input_ids"])
ref_toks   = dummy["input_tokens"]

# Everything below is derived from the reference itself. This tokenizer's
# special-token attributes disagree with the checkpoint: mask_token_id says 4
# but the real mask is 35107, and pad_token_id does not equal the padding value
# actually used here - so nothing may be taken from those attributes.
#
# The rows are padded to equal length, and input_tokens spells the padding out,
# so "<pad>" occurrences give the exact padding width per row.
def unpadded(row, tok_str):
    n_pad = tok_str.count("<pad>")
    return row[:len(row) - n_pad] if n_pad else list(row)

# A row with no padding ends exactly at </AMR>, so its last four ids are the
# suffix: </s> <AMR> <mask> </AMR>
i_full     = next(i for i, s in enumerate(ref_toks) if "<pad>" not in s)
AMR_SUFFIX = list(ref_ids[i_full])[-4:]
PAD_ID     = ref_ids[0][-1]          # trailing id of a padded row

print("suffix ids        :", AMR_SUFFIX, "(expect [2, 38023, 35107, 38024])")
print("pad id            :", PAD_ID)
print("vocab <mask>      :", amr_tokenizer.vocab["<mask>"])
print("attribute mask id :", amr_tokenizer.mask_token_id, "(ignored)")
print("attribute pad id  :", amr_tokenizer.pad_token_id, "(ignored)")
print("")


def build_input_ids(text):
    """Plain Indonesian text -> the exact input this checkpoint expects."""
    ids = amr_tokenizer(text, max_length=None, truncation=True,
                        add_special_tokens=False)["input_ids"]
    return ids + AMR_SUFFIX


SUFFIX_STR = "</s> <AMR> <mask> </AMR>"
n_pass = 0
for tok_str, ref in zip(ref_toks, ref_ids):
    text = tok_str.replace("<pad>", "").strip()
    if SUFFIX_STR in text:
        text = text[:text.index(SUFFIX_STR)]
    text = text.replace("</s>", "").strip()

    expected = unpadded(ref, tok_str)
    ours     = build_input_ids(text)
    ok = ours == expected
    n_pass += ok
    print(("PASS  " if ok else "FAIL  ") + repr(text))
    if not ok:
        print("   expected:", expected)
        print("   ours    :", ours)

print("")
print(str(n_pass) + "/" + str(len(ref_ids)) + " matched the model's reference input.")
assert n_pass == len(ref_ids), (
    "Input format mismatch - fix build_input_ids before running the full job.")

# 5. Carry forward the previous run

Copies the previous run's graphs into `/kaggle/working` **before** parsing, so
a wrong `PREV_DIR` is caught now rather than after hours of work, and so an
interrupted session still saves a cumulative output.

On the first run set `PREV_DIR = ""`; 0 copied is expected.

In [ ]:
# Carry forward FIRST, before any parsing.
#
# /kaggle/working starts empty every session, so the previous run's graphs are
# copied in now rather than at the end. Two reasons:
#   1. it validates PREV_DIR immediately - a wrong path shows up here as 0
#      copied, instead of after hours of parsing;
#   2. if this session dies or hits Kaggle's wall mid-parse, whatever gets saved
#      is still CUMULATIVE. Copying at the end means a killed run saves only the
#      new graphs and the resume chain breaks.
copied = 0
if PREV_DIR and os.path.isdir(PREV_DIR):
    have = set(os.listdir(OUTPUT_DIR))
    for fn in tqdm(os.listdir(PREV_DIR), desc="Carrying forward"):
        if fn.endswith(".txt") and fn not in have:
            shutil.copyfile(os.path.join(PREV_DIR, fn), os.path.join(OUTPUT_DIR, fn))
            copied += 1
else:
    print("PREV_DIR not found (fine on the first run): " + str(PREV_DIR))

carried_total = len([f for f in os.listdir(OUTPUT_DIR) if f.endswith(".txt")])
print("Carried forward   : " + str(copied))
print("Now in output dir : " + str(carried_total))

# 6. Dataset & DataLoader

In [ ]:
class Liputan6Dataset(Dataset):
    """One row per SENTENCE (id = '{doc_id}_{sent_idx}').

    Keeps only sentences not yet parsed, looking at both this run's output and
    the previous run's mounted output so the job resumes across sessions.
    """
    def __init__(self, data_path=DATA_PATH, output_dir=OUTPUT_DIR,
                 doc_limit=DOC_LIMIT, sent_limit=SENT_LIMIT, split=SPLIT):
        df = pd.read_csv(os.path.join(data_path, "analysis_data.csv"), dtype={"id": str})
        total_rows, total_docs = len(df), df["doc_id"].nunique()

        if split is not None:
            if "split" not in df.columns:
                raise KeyError("csv has no 'split' column - regenerate it with "
                               "liputan6_to_csv.py so train/test are not mixed")
            print("Available splits    :", df["split"].value_counts().to_dict())
            df = df[df["split"] == split]
            if df.empty:
                raise ValueError(f"no rows for split={split!r}")

        if doc_limit is not None:
            keep = df["doc_id"].drop_duplicates().head(doc_limit)
            df = df[df["doc_id"].isin(set(keep))]

        if sent_limit is not None:
            # take whole documents, in order, until the sentence budget is spent
            sizes = df.groupby("doc_id", sort=False).size()
            keep = sizes.cumsum().le(sent_limit)
            keep_ids = set(sizes.index[keep])
            if not keep_ids:                 # first doc alone exceeds the budget
                keep_ids = {sizes.index[0]}
            df = df[df["doc_id"].isin(keep_ids)]

        parsed = set()
        for d in [PREV_DIR, output_dir]:
            if d and os.path.isdir(d):
                for fn in os.listdir(d):
                    if fn.endswith(".txt"):
                        parsed.add(fn[:-4])

        in_scope = len(df)
        self.df = df[~df["id"].isin(parsed)].reset_index(drop=True)

        print(f"Corpus              : {total_rows} sentences / {total_docs} docs")
        print(f"In scope ({split}): {in_scope} sentences / "
              f"{df['doc_id'].nunique()} docs  "
              f"[doc_limit={doc_limit}, sent_limit={sent_limit}]")
        print(f"Already parsed      : {in_scope - len(self.df)}")
        print(f"Remaining to parse  : {len(self.df)}")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {"id": row["id"], "text": str(row["text"])}


def collate_fn(batch):
    """Plain Indonesian text -> ids + </s> <AMR> <mask> </AMR>, then padded.

    Format verified against the model's dummy_input.json in section 4.
    PAD_ID comes from that reference, not from amr_tokenizer.pad_token_id,
    which disagrees with this checkpoint.
    """
    ids   = [b["id"] for b in batch]
    built = [build_input_ids(b["text"]) for b in batch]
    width = max(len(x) for x in built)
    input_ids = [x + [PAD_ID] * (width - len(x)) for x in built]
    masks     = [[1] * len(x) + [0] * (width - len(x)) for x in built]
    return ids, input_ids, masks


ds     = Liputan6Dataset()
loader = DataLoader(ds, batch_size=BATCH_SIZE, collate_fn=collate_fn)

# 7. Decoding & storage

In [ ]:
# The model CONFIG is reliable here (bos 0, pad 1, eos 2 - all agree with
# dummy_input.json); the tokenizer's special-token attributes are not, so read
# these from the config and from the verified suffix.
BOS_ID  = amr_config.bos_token_id
EOS_ID  = amr_config.eos_token_id
AMR_EOS = amr_tokenizer.vocab["</AMR>"]
AMR_BOS = amr_tokenizer.vocab["<AMR>"]


def decode_amr_output(pred_token_ids, tokenizer):
    pred_ids = list(pred_token_ids)
    pred_ids[0] = BOS_ID
    pred_ids = [EOS_ID if tok == AMR_EOS else tok
                for tok in pred_ids if tok != PAD_ID]
    graph, status, _ = tokenizer.decode_amr(pred_ids, restore_name_ops=False)
    return penman.encode(graph), status


def store_graph(ids, amr_strings, output_dir=OUTPUT_DIR):
    for data_id, amr_str in zip(ids, amr_strings):
        with open(os.path.join(output_dir, f"{data_id}.txt"), "w", encoding="utf-8") as f:
            f.write(amr_str)

# 7b. VERIFY decoding

Section 4 checked the input side. This checks the output side by decoding the
reference `labels` from `dummy_input.json` - gold AMR for sentences we know -
through the exact path used at inference.

It matters because `decode_amr` reads `amr_tokenizer.vocab`, whose
special-token region is off by one against this checkpoint. If decoding is
wrong, every graph the notebook writes is wrong in the same way, and nothing
raises an error.

In [ ]:
# Decode-side check. Section 4 proved we ENCODE what the checkpoint expects;
# this proves we DECODE its output correctly. decode_amr() reads the same
# amr_tokenizer.vocab whose special-token region is off by one, so it could be
# wrong even though the AMR token range (37xxx-38xxx) is intact.
#
# dummy_input.json ships reference `labels` - gold AMR token ids for the same
# five sentences - so we can decode known-good output and see what comes out.
ref_labels = ast.literal_eval(dummy["labels"])
AMR_BOS    = AMR_SUFFIX[1]

n_ok = 0
for text_str, label_row in zip(ref_toks, ref_labels):
    snt = text_str.replace("<pad>", "").strip()
    if SUFFIX_STR in snt:
        snt = snt[:snt.index(SUFFIX_STR)]
    snt = snt.replace("</s>", "").strip()

    ids = [t for t in label_row if t != PAD_ID]
    try:
        # generate() emits decoder_start_token_id first, so mimic that shape
        amr_str, status = decode_amr_output([AMR_BOS] + ids, amr_tokenizer)
        name = getattr(status, "name", str(status))
        good = name in ("OK", "FIXED")
        n_ok += good
        print(("PASS  " if good else "WARN  ") + repr(snt) + "   status=" + name)
        for line in amr_str.split("\n"):
            print("      " + line)
    except Exception as exc:
        print("FAIL  " + repr(snt) + "   " + type(exc).__name__ + ": " + str(exc))
    print("")

print(str(n_ok) + "/" + str(len(ref_labels)) + " reference graphs decoded cleanly.")
print("")
print("Read the graphs above - they should describe the Indonesian sentence")
print("(e.g. 'Coba lihat.' -> a look/see predicate, imperative mode).")
print("Garbled concepts mean decode_amr is mis-reading the vocab, and every")
print("graph this notebook produces would be wrong in the same way.")

# 8. Smoke test - parse 3 sentences and look at them

Read these before launching the full run. The concepts should reflect the
sentence's meaning and the status should be `OK`; a `BACKOFF` means the decoder
produced something unparseable.

In [ ]:
import time as _t
_n = min(3, len(ds))
_start = _t.time()
for i in range(_n):
    sample = ds[i]
    _, inp, msk = collate_fn([sample])
    with torch.no_grad():
        out = amr_model.generate(
            input_ids=torch.tensor(inp, dtype=torch.long).to(device),
            attention_mask=torch.tensor(msk, dtype=torch.long).to(device),
            num_beams=NUM_BEAMS, max_length=MAX_GEN_LEN,
            decoder_start_token_id=AMR_BOS,
            eos_token_id=AMR_EOS, pad_token_id=PAD_ID)
    amr_str, status = decode_amr_output(out[0].cpu().tolist(), amr_tokenizer)
    print(f"--- {sample['id']} | status={getattr(status, 'name', status)}")
    print(sample["text"])
    print(amr_str)
    print()

_elapsed = _t.time() - _start
print("--- timing ---")
print("sentences      : " + str(_n))
print("seconds each   : " + format(_elapsed / max(_n, 1), ".2f"))
_rate = _elapsed / max(_n, 1)
print("projected full : " + format(_rate * len(ds) / 3600, ".2f") + " h" + " for " + str(len(ds)) + " sentences (batch_size=1 here)")
print("note: section 8 uses BATCH_SIZE=" + str(BATCH_SIZE) + ", so expect meaningfully faster")

# 9. Parse everything in scope

In [ ]:
start_t  = time.time()
budget_s = TIME_BUDGET_H * 3600
status_counts = {"OK": 0, "FIXED": 0, "BACKOFF": 0, "ERROR": 0}
total_parsed, stop_reason = 0, "completed all remaining"
print(f"Parsing {len(ds)} sentences...")

pbar = tqdm(loader, desc="Parsing AMR")
for batch_ids, inputs, masks in pbar:
    if MAX_NEW is not None and total_parsed >= MAX_NEW:
        stop_reason = f"hit MAX_NEW={MAX_NEW}"
        break
    if time.time() - start_t > budget_s:
        stop_reason = f"hit TIME_BUDGET_H={TIME_BUDGET_H}"
        break
    try:
        with torch.no_grad():
            outputs = amr_model.generate(
                input_ids=torch.tensor(inputs, dtype=torch.long).to(device),
                attention_mask=torch.tensor(masks, dtype=torch.long).to(device),
                num_beams=NUM_BEAMS, max_length=MAX_GEN_LEN,
                decoder_start_token_id=AMR_BOS,
                # This checkpoint ends a graph with </AMR> (38024), not with
                # config.eos_token_id (2). Without this, generate() never sees
                # its stop token and runs to max_length on EVERY sentence -
                # the model card's mean gen_len is 27.5, so that is ~19x wasted.
                eos_token_id=AMR_EOS,
                pad_token_id=PAD_ID)
        amr_strings = []
        for i in range(outputs.shape[0]):
            amr_string, status = decode_amr_output(outputs[i].cpu().tolist(), amr_tokenizer)
            amr_strings.append(amr_string)
            name = getattr(status, "name", str(status))
            status_counts[name if name in status_counts else "ERROR"] += 1
        store_graph(batch_ids, amr_strings)
        total_parsed += len(batch_ids)
    except Exception:
        status_counts["ERROR"] += len(batch_ids)
        continue
    if total_parsed % 500 == 0:
        torch.cuda.empty_cache()
    pbar.set_postfix(new=total_parsed, h=f"{(time.time()-start_t)/3600:.2f}")

pbar.close()
print("")
print(f"Stopped because : {stop_reason}")
print(f"Parsed this run : {total_parsed}")
print(f"Elapsed         : {(time.time()-start_t)/3600:.2f} h")
for k, v in status_counts.items():
    if v:
        print(f"  {k:8s}: {v} ({v/max(total_parsed,1)*100:.1f}%)")

# 10. Zip for download (optional)

Only needed if you want the graphs on your own machine. To build a Kaggle
dataset, use the notebook output's `amr_graphs/` folder directly and skip this
- otherwise the dataset ends up holding two copies of every graph.

In [ ]:
import zipfile

# Store files under an "amr_graphs/" prefix so that a dataset built from this
# zip mounts at the SAME path as one built from the notebook output. Without the
# prefix Kaggle extracts the .txt files to the dataset root and PREV_DIR has to
# change depending on how the dataset was made.
zip_path = "/kaggle/working/liputan6_amr_graphs.zip"
files = [f for f in os.listdir(OUTPUT_DIR) if f.endswith(".txt")]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in tqdm(files, desc="Zipping"):
        zf.write(os.path.join(OUTPUT_DIR, fn), os.path.join("amr_graphs", fn))
print(f"{zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB), {len(files)} graphs")
print("")
print("Either route gives the same layout, so PREV_DIR stays:")
print("  " + BASE + "/liputan6-amr-graphs/amr_graphs")
print("")
print("The zip is only worth keeping if you want to DOWNLOAD the graphs. If you")
print("are making a Kaggle dataset from the notebook output, delete it first so")
print("the dataset does not end up with two copies of every graph:")
print("  os.remove(zip_path)")